# CUDA Denoise + Edge Detection — Colab Test

**IMPORTANT:** Before running, enable GPU runtime:
> `Runtime` → `Change runtime type` → Hardware accelerator: **T4 GPU** → Save

This notebook tests the CUDA implementation of the HPC image processing project:
1. Gaussian blur (denoising)
2. Sobel edge detection

Both are parallelized as CUDA kernels — one GPU thread per pixel.

In [ ]:
# Step 1: Verify GPU is available
!nvidia-smi
!echo '---'
!nvcc --version

In [ ]:
# Step 2: Install C++ OpenCV (takes ~30 seconds)
!apt-get update -qq
!apt-get install -y -qq libopencv-dev
!pkg-config --modversion opencv4 && echo 'OpenCV ready'

In [ ]:
# Step 3: Write the CUDA source file
cuda_src = '''
#include <opencv2/opencv.hpp>
#include <iostream>
#include <chrono>
#include <cuda_runtime.h>

using namespace std;
using namespace cv;
using namespace std::chrono;

__constant__ float d_gauss[9] = {
    1/16.0f, 2/16.0f, 1/16.0f,
    2/16.0f, 4/16.0f, 2/16.0f,
    1/16.0f, 2/16.0f, 1/16.0f
};

__constant__ int d_sobel_x[9] = {-1, 0, 1, -2, 0, 2, -1, 0, 1};
__constant__ int d_sobel_y[9] = {-1, -2, -1,  0, 0, 0,  1,  2,  1};

__global__ void gaussianBlurKernel(const unsigned char* input, unsigned char* output, int rows, int cols)
{
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= rows || j >= cols) return;
    if (i == 0 || i == rows - 1 || j == 0 || j == cols - 1) {
        output[i * cols + j] = input[i * cols + j];
        return;
    }
    float sum = 0.0f;
    for (int ki = -1; ki <= 1; ki++) {
        for (int kj = -1; kj <= 1; kj++) {
            sum += input[(i + ki) * cols + (j + kj)] * d_gauss[(ki + 1) * 3 + (kj + 1)];
        }
    }
    output[i * cols + j] = (unsigned char)min(max((int)sum, 0), 255);
}

__global__ void sobelEdgeKernel(const unsigned char* input, unsigned char* output, int rows, int cols)
{
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= rows || j >= cols) return;
    if (i == 0 || i == rows - 1 || j == 0 || j == cols - 1) {
        output[i * cols + j] = 0;
        return;
    }
    int gx = 0, gy = 0;
    for (int ki = -1; ki <= 1; ki++) {
        for (int kj = -1; kj <= 1; kj++) {
            unsigned char pixel = input[(i + ki) * cols + (j + kj)];
            gx += pixel * d_sobel_x[(ki + 1) * 3 + (kj + 1)];
            gy += pixel * d_sobel_y[(ki + 1) * 3 + (kj + 1)];
        }
    }
    int mag = (int)sqrtf((float)(gx * gx + gy * gy));
    output[i * cols + j] = (unsigned char)min(mag, 255);
}

int main(int argc, char** argv)
{
    if (argc < 2) {
        cerr << "Usage: ./cuda_denoise_edge <image_path>" << endl;
        return 1;
    }
    string imagePath = argv[1];
    Mat img = imread(imagePath, IMREAD_GRAYSCALE);
    if (img.empty()) {
        cerr << "Failed to load image" << endl;
        return 1;
    }
    int rows = img.rows;
    int cols = img.cols;
    size_t imgSize = (size_t)rows * cols * sizeof(unsigned char);

    unsigned char *d_input, *d_denoised, *d_edges;
    cudaMalloc(&d_input, imgSize);
    cudaMalloc(&d_denoised, imgSize);
    cudaMalloc(&d_edges, imgSize);
    cudaMemcpy(d_input, img.data, imgSize, cudaMemcpyHostToDevice);

    dim3 blockSize(16, 16);
    dim3 gridSize((cols + blockSize.x - 1) / blockSize.x,
                  (rows + blockSize.y - 1) / blockSize.y);

    auto start = high_resolution_clock::now();
    gaussianBlurKernel<<<gridSize, blockSize>>>(d_input, d_denoised, rows, cols);
    cudaDeviceSynchronize();
    sobelEdgeKernel<<<gridSize, blockSize>>>(d_denoised, d_edges, rows, cols);
    cudaDeviceSynchronize();
    auto end = high_resolution_clock::now();
    double elapsed = duration<double, milli>(end - start).count();

    Mat denoised(rows, cols, CV_8UC1);
    Mat edges(rows, cols, CV_8UC1);
    cudaMemcpy(denoised.data, d_denoised, imgSize, cudaMemcpyDeviceToHost);
    cudaMemcpy(edges.data, d_edges, imgSize, cudaMemcpyDeviceToHost);

    cudaFree(d_input);
    cudaFree(d_denoised);
    cudaFree(d_edges);

    imwrite("cuda_denoised.png", denoised);
    imwrite("cuda_edges.png", edges);

    cout << "CUDA execution time: " << elapsed << " ms" << endl;
    cout << "Image size: " << cols << "x" << rows << endl;
    return 0;
}
'''

with open('denoise_cuda.cu', 'w') as f:
    f.write(cuda_src.strip())
print('CUDA source written to denoise_cuda.cu')

In [ ]:
# Step 4: Compile with nvcc + OpenCV
!nvcc -O2 denoise_cuda.cu -o cuda_denoise_edge \
    -I/usr/include/opencv4 \
    -lopencv_core -lopencv_imgproc -lopencv_imgcodecs \
    2>&1
!ls -lh cuda_denoise_edge && echo 'Compilation successful!'

In [ ]:
# Step 5: Generate a synthetic noisy test image
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

np.random.seed(42)

# Create a structured base image (checkerboard + circles)
size = 512
img = np.zeros((size, size), dtype=np.uint8)

# Horizontal and vertical bands
for i in range(0, size, 64):
    img[i:i+32, :] = 180
for j in range(0, size, 64):
    img[:, j:j+32] = np.clip(img[:, j:j+32].astype(int) + 60, 0, 255).astype(np.uint8)

# Add Gaussian noise
noise = np.random.normal(0, 25, (size, size))
noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

# Save
Image.fromarray(noisy).save('test_noisy.png')

# Preview
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Clean')
axes[1].imshow(noisy, cmap='gray')
axes[1].set_title('Noisy (input for CUDA)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Test image saved: {size}x{size} pixels')

In [ ]:
# Step 6: Run the CUDA binary
!./cuda_denoise_edge test_noisy.png

In [ ]:
# Step 7: Verify output files were created
import os
for f in ['cuda_denoised.png', 'cuda_edges.png']:
    size_kb = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  {f}: {status} ({size_kb:.1f} KB)')

In [ ]:
# Step 8: Display results
import cv2
import numpy as np
import matplotlib.pyplot as plt

original = cv2.imread('test_noisy.png', cv2.IMREAD_GRAYSCALE)
denoised = cv2.imread('cuda_denoised.png', cv2.IMREAD_GRAYSCALE)
edges    = cv2.imread('cuda_edges.png',    cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
images = [original, denoised, edges]
titles = ['Input (Noisy)', 'CUDA Denoised\n(Gaussian Blur)', 'CUDA Edges\n(Sobel)']
cmaps  = ['gray', 'gray', 'hot']

for ax, img, title, cmap in zip(axes, images, titles, cmaps):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.suptitle('CUDA GPU Image Processing — Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('CUDA verification complete!')

In [ ]:
# Step 9 (optional): Quick RMSE check vs serial-equivalent Python baseline
from scipy.ndimage import gaussian_filter
import numpy as np

original = cv2.imread('test_noisy.png', cv2.IMREAD_GRAYSCALE).astype(np.float32)
cuda_out  = cv2.imread('cuda_denoised.png', cv2.IMREAD_GRAYSCALE).astype(np.float32)

# Serial Gaussian blur (sigma chosen to approximate 3x3 kernel)
serial_blur = gaussian_filter(original, sigma=0.85)

rmse = np.sqrt(np.mean((cuda_out - serial_blur) ** 2))
print(f'RMSE between CUDA denoised and Python gaussian_filter: {rmse:.4f}')
print('(Small RMSE means CUDA kernel is numerically correct)')

## Compilation command for the real project (WSL/Linux)

Once you have GPU access (e.g. a machine with NVIDIA GPU + CUDA toolkit installed), compile with:

```bash
cd cuda/
nvcc -O2 denoise_cuda.cu -o cuda_denoise_edge \
    $(pkg-config --cflags --libs opencv4) 
```

Then run:
```bash
./cuda_denoise_edge /path/to/image.png
# Output: cuda_denoised.png, cuda_edges.png
```